# 第四章：分离数据和指令

- [课程](#lesson)
- [练习](#exercises)
- [示例实验区](#example-playground)

## 设置

运行以下设置单元格以加载您的 API 密钥并建立 `get_completion` 辅助函数。

In [ ]:
# Import python's built-in regular expression library
import re
import boto3
import json

# Import the hints module from the utils package
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Retrieve the MODEL_NAME variable from the IPython store
%store -r MODEL_NAME
%store -r AWS_REGION

client = boto3.client('bedrock-runtime',region_name=AWS_REGION)

def get_completion(prompt,system=''):
    body = json.dumps(
        {
            "anthropic_version": '',
            "max_tokens": 2000,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.0,
            "top_p": 1,
            "system": system
        }
    )
    response = client.invoke_model(body=body, modelId=MODEL_NAME)
    response_body = json.loads(response.get('body').read())

    return response_body.get('content')[0].get('text')

---

## 课程

通常，我们不想编写完整的提示，而是想要**可以在提交给 Claude 之前使用额外输入数据修改的提示模板**。如果您希望 Claude 每次执行相同的操作，但 Claude 用于任务的数据每次可能不同，这可能会很方便。

幸运的是，我们可以通过**将提示的固定框架与可变用户输入分离，然后在将完整提示发送给 Claude 之前将用户输入替换到提示中**来轻松做到这一点。

下面，我们将逐步介绍如何编写可替换的提示模板，以及如何替换用户输入。

### 示例

在第一个例子中，我们要求 Claude 充当动物声音生成器。请注意，提交给 Claude 的完整提示只是用输入（在本例中为 "Cow"）替换的 `PROMPT_TEMPLATE`。请注意，当我们打印出完整提示时，"Cow" 一词通过 f-string 替换了 `ANIMAL` 占位符。

**注意：** 在实践中，您不必将占位符变量命名为任何特定名称。我们在这个例子中称其为 `ANIMAL`，但同样容易，我们可以称其为 `CREATURE` 或 `A`（尽管通常最好让您的变量名称具体且相关，以便您的提示模板即使没有替换也易于理解，只是为了用户的可解析性）。只需确保您为变量命名的内容是您用于提示模板 f-string 的内容。

In [ ]:
# Variable content
ANIMAL = "Cow"

# Prompt template with a placeholder for the variable content
PROMPT = f"I will tell you the name of an animal. Please respond with the noise that animal makes. {ANIMAL}"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

为什么我们要这样分离和替换输入？嗯，**提示模板简化了重复性任务**。假设您构建了一个提示结构，邀请第三方用户向提示提交内容（在本例中是他们想要生成声音的动物）。这些第三方用户不必编写甚至查看完整的提示。他们所要做的就是填写变量。

我们在这里使用变量和 f-string 进行替换，但您也可以使用 format() 方法来完成。

**注意：** 提示模板可以根据需要包含任意数量的变量！

当引入这样的替换变量时，非常重要的是**确保 Claude 知道变量的开始和结束位置**（与指令或任务描述相比）。让我们看一个例子，其中指令和替换变量之间没有分隔。

对于我们的人眼来说，下面的提示模板中变量的开始和结束位置非常清楚。但是，在完全替换的提示中，这种界限变得不清楚。

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

在这里，**Claude 认为 "Yo Claude" 是它应该重写的电子邮件的一部分**！您可以看出来，因为它以 "Dear Claude" 开始重写。对于人眼来说，特别是在提示模板中，电子邮件的开始和结束位置很清楚，但在替换后的提示中就变得不那么清楚了。

我们如何解决这个问题？**用 XML 标签包装输入**！我们在下面这样做了，正如您所看到的，输出中不再有 "Dear Claude"。

[XML 标签](https://docs.anthropic.com/claude/docs/use-xml-tags)是像 `<tag></tag>` 这样的尖括号标签。它们成对出现，由开始标签（如 `<tag>`）和由 `/` 标记的结束标签（如 `</tag>`）组成。XML 标签用于包装内容，如下所示：`<tag>content</tag>`。

**注意：** 虽然 Claude 可以识别和处理各种分隔符和定界符，但我们建议您**专门使用 XML 标签作为 Claude 的分隔符**，因为 Claude 经过专门训练，可以将 XML 标签识别为提示组织机制。除了函数调用之外，**没有 Claude 经过训练的特殊 XML 标签可以最大限度地提高您的性能**。我们有意使 Claude 以这种方式非常具有可塑性和可定制性。

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

让我们看另一个 XML 标签如何帮助我们的例子。

在以下提示中，**Claude 错误地解释了提示的哪一部分是指令与输入**。由于格式的原因，它错误地将 `Each is about an animal, like rabbits` 视为列表的一部分，而用户（填写 `SENTENCES` 变量的人）可能不希望这样。

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

要解决这个问题，我们只需要**用 XML 标签包围用户输入的句子**。这向 Claude 显示输入数据的开始和结束位置，尽管 `Each is about an animal, like rabbits.` 前面有误导性的连字符。

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f""" Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
<sentences>
{SENTENCES}
</sentences>"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

**注意：** 在 "Each is about an animal" 提示的错误版本中，我们必须包含连字符才能让 Claude 以我们想要的方式对此示例做出错误响应。这是关于提示的重要一课：**小细节很重要**！**仔细检查您的提示是否有拼写错误和语法错误**总是值得的。Claude 对模式很敏感（在早期，在微调之前，它是一个原始的文本预测工具），当您犯错误时，它更容易犯错误，当您听起来聪明时更聪明，当您听起来愚蠢时更愚蠢，等等。

如果您想在不更改上面任何内容的情况下尝试课程提示，请滚动到课程 notebook 的最底部访问[**示例实验区**](#example-playground)。

---

## 练习
- [练习 4.1 - 俳句主题](#exercise-41---haiku-topic)
- [练习 4.2 - 狗问题与拼写错误](#exercise-42---dog-question-with-typos)
- [练习 4.3 - 狗问题第二部分](#exercise-42---dog-question-part-2)

### 练习 4.1 - 俳句主题
修改 `PROMPT`，使其成为一个模板，该模板将接受一个名为 `TOPIC` 的变量并输出关于该主题的俳句。此练习只是为了测试您对使用 f-string 的变量模板结构的理解。

In [ ]:
# Variable content
TOPIC = "Pigs"

# Prompt template with a placeholder for the variable content
PROMPT = f""

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("pigs", text.lower()) and re.search("haiku", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ 如果您需要提示，请运行下面的单元格！

In [ ]:
print(hints.exercise_4_1_hint)

### 练习 4.2 - 狗问题与拼写错误
通过添加 XML 标签来修复 `PROMPT`，以便 Claude 产生正确的答案。

尽量不要改变提示的其他任何内容。混乱和错误百出的写作是有意为之的，这样您就可以看到 Claude 对此类错误的反应。

In [ ]:
# Variable content
QUESTION = "ar cn brown?"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ 如果您需要提示，请运行下面的单元格！

In [ ]:
print(hints.exercise_4_2_hint)

### 练习 4.3 - 狗问题第二部分
修复 `PROMPT`，**不添加** XML 标签。相反，只从提示中删除一两个单词。

就像上面的练习一样，尽量不要改变提示的其他任何内容。这将向您展示 Claude 可以解析和理解什么样的语言。

In [ ]:
# Variable content
QUESTION = "ar cn brown?"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ 如果您需要提示，请运行下面的单元格！

In [ ]:
print(hints.exercise_4_3_hint)

### 恭喜！

如果您已经解决了到目前为止的所有练习，那么您就可以进入下一章了。祝您提示愉快！

---

## 示例实验区

这是一个让您自由尝试本课程中显示的提示示例并调整提示以查看它如何影响 Claude 响应的区域。

In [ ]:
# Variable content
ANIMAL = "Cow"

# Prompt template with a placeholder for the variable content
PROMPT = f"I will tell you the name of an animal. Please respond with the noise that animal makes. {ANIMAL}"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f""" Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
<sentences>
{SENTENCES}
</sentences>"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))